# Ch. 2 · TP2 Qualité des données

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/benlahmar/IASciences/blob/main/notebooks/ch02-tp2-qualite-donnees.ipynb)

**IA pour les sciences — M1.** Valeurs manquantes, valeurs aberrantes et standardisation : décider sans fabriquer des observations.

**Durée indicative :** 3 h

**Objectifs**

- Distinguer les mécanismes de données manquantes (MCAR, MNAR) et leurs conséquences.
- Détecter des valeurs atypiques par z-score, IQR et score robuste (médiane/MAD).
- Mesurer l’effet d’une imputation sur la moyenne, la dispersion et la corrélation.
- Standardiser sans fuite d’information.

**Prérequis :** TP1 du chapitre 2.

> **Mode d’emploi.** Exécutez les cellules dans l’ordre. Les cellules marquées **✏️ Exercice** contiennent des `...` à compléter : une cellule non complétée lève une erreur, c’est normal. Les corrigés sont repliés sous **« Voir le corrigé »** : cherchez d’abord ! Les cellules suivantes ne dépendent jamais du résultat d’un exercice.

## 0. Données de départ (reprises du TP1)

Ce notebook est autonome : les données simulées du TP1 sont recréées.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n = 60
df = pd.DataFrame({
    'echantillon': [f'E{i:03d}' for i in range(n)],
    'temperature_C': rng.uniform(15, 35, n).round(1),
})
df['concentration_mg_L'] = (0.4 * df['temperature_C'] + rng.normal(0, 2, n)).round(2)
col = 'concentration_mg_L'
print(df[col].describe().round(2))

## 1. Pourquoi une valeur manque-t-elle ?

Le mécanisme compte plus que le nombre de valeurs manquantes :

| Mécanisme | Exemple | Conséquence si on supprime les lignes |
|---|---|---|
| **MCAR** (complètement au hasard) | tube cassé au hasard | perte de précision, pas de biais |
| **MAR** (dépend d’une variable observée) | capteur défaillant aux fortes températures | biais possible, corrigeable avec la variable observée |
| **MNAR** (dépend de la valeur elle-même) | concentration **sous la limite de détection** non reportée | **biais** : la moyenne est surestimée |

**Démonstration.** Nous retirons 25 % des valeurs de deux façons et comparons la moyenne des valeurs restantes à la vraie moyenne.

In [ ]:
vraie_moyenne = df[col].mean()
k = int(0.25 * n)

mcar = df[col].copy()
mcar[rng.choice(n, size=k, replace=False)] = np.nan

mnar = df[col].copy()
mnar[df[col].nsmallest(k).index] = np.nan   # les plus faibles valeurs « disparaissent » (limite de détection)

print(f'Vraie moyenne            : {vraie_moyenne:.2f} mg/L')
print(f'Moyenne observée (MCAR)  : {mcar.mean():.2f} mg/L')
print(f'Moyenne observée (MNAR)  : {mnar.mean():.2f} mg/L  <- biais systématique')

## 2. Un jeu avec manquants et une valeur suspecte

In [ ]:
df2 = df.copy()
df2.loc[[3, 14, 22, 40], col] = np.nan
df2.loc[8, col] = 65.0          # saisie suspecte (6,5 tapé 65 ?)
print(df2.isna().sum())
print(df2[col].describe().round(2))

## 3. Trois détecteurs de valeurs atypiques

- **z-score** : $z_i = (x_i - \bar{x})/s$ ; on signale $|z|>3$. Problème : la valeur aberrante gonfle elle-même $\bar{x}$ et $s$ (*effet de masque*).
- **Règle de Tukey (IQR)** : on signale $x < Q_1 - 1{,}5\,IQR$ ou $x > Q_3 + 1{,}5\,IQR$.
- **Score robuste** : $z^{rob}_i = 0{,}6745\,(x_i - \text{médiane})/\text{MAD}$, avec $\text{MAD}=\text{médiane}(|x_i-\text{médiane}|)$ ; on signale $|z^{rob}|>3{,}5$.

Signaler n’est **pas** supprimer : on vérifie la fiche de laboratoire.

In [ ]:
x = df2[col]
z = (x - x.mean()) / x.std()
mediane = x.median()
mad = (x - mediane).abs().median()
z_rob = 0.6745 * (x - mediane) / mad
q1, q3 = x.quantile([.25, .75])
iqr = q3 - q1

table = pd.DataFrame({'valeur': x, 'z': z.round(2), 'z_robuste': z_rob.round(2)})
table['tukey'] = (x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)
print(table.loc[table['tukey'] | (z.abs() > 3) | (z_rob.abs() > 3.5)])
print(f'Moyenne {x.mean():.2f} vs médiane {mediane:.2f} : la moyenne est tirée par la valeur 65.')

## 4. Imputer et mesurer les conséquences

L’imputation par la médiane remplit les trous, mais **réduit artificiellement la dispersion** et **affaiblit les corrélations**
(on a ajouté des points « plats »). On compare en excluant d’abord la valeur suspecte (après vérification, supposons une erreur de saisie).

In [ ]:
df_verif = df2.copy()
df_verif.loc[8, col] = np.nan     # erreur de saisie confirmée -> traitée comme manquante, documentée

df_median = df_verif.copy()
df_median[col] = df_median[col].fillna(df_verif[col].median())

print('Écart-type (valeurs présentes) :', round(df_verif[col].std(), 3))
print('Écart-type (après imputation)  :', round(df_median[col].std(), 3))
print('Corrélation (valeurs présentes):', round(df_verif[['temperature_C', col]].corr().iloc[0, 1], 3))
print('Corrélation (après imputation) :', round(df_median[['temperature_C', col]].corr().iloc[0, 1], 3))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (nom, serie) in zip(axes, [('Valeurs présentes', df_verif[col].dropna()),
                                   ('Imputation médiane', df_median[col])]):
    ax.hist(serie, bins=10, edgecolor='white')
    ax.set(title=nom, xlabel='Concentration (mg/L)', ylabel='Effectif')
plt.tight_layout(); plt.show()

### ✏️ Exercice 1 — Imputation par régression

Imputez les valeurs manquantes à partir de la température (droite ajustée sur les lignes complètes). Comparez écart-type et corrélation avec l’imputation médiane. Quel risque introduit cette méthode ?

In [ ]:
complet = df_verif.dropna(subset=[col])
pente, origine = np.polyfit(..., ..., 1)
df_reg = df_verif.copy()
manquant = df_reg[col].isna()
df_reg.loc[manquant, col] = ...
print('Écart-type :', round(df_reg[col].std(), 3))
print('Corrélation :', round(df_reg[['temperature_C', col]].corr().iloc[0, 1], 3))

<details><summary><b>Voir le corrigé</b></summary>

```python
complet = df_verif.dropna(subset=[col])
pente, origine = np.polyfit(complet['temperature_C'], complet[col], 1)
df_reg = df_verif.copy()
manquant = df_reg[col].isna()
df_reg.loc[manquant, col] = origine + pente * df_reg.loc[manquant, 'temperature_C']
print('Écart-type :', round(df_reg[col].std(), 3))
print('Corrélation :', round(df_reg[['temperature_C', col]].corr().iloc[0, 1], 3))
```

Les valeurs imputées tombent exactement sur la droite : la corrélation est **gonflée** (on a ajouté des points sans bruit). Si l’on modélise ensuite la concentration à partir de la température, on évalue le modèle sur des valeurs qu’il a lui-même fabriquées. Toute imputation doit être déclarée, et pour un objectif prédictif, apprise uniquement sur l’entraînement.

</details>

## 5. Standardiser sans fuite

$z = (x-\mu)/\sigma$. Pour un objectif **prédictif**, $\mu$ et $\sigma$ doivent être estimés sur l’entraînement seulement,
puis appliqués au test. Sinon, le test « informe » la préparation des données.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[['temperature_C', 'concentration_mg_L']]
X_train, X_test = train_test_split(X, test_size=0.3, random_state=0)
scaler = StandardScaler().fit(X_train)          # apprend mu et sigma sur l'entraînement
print('mu appris   :', scaler.mean_.round(2))
print('sigma appris:', scaler.scale_.round(2))
Z_test = scaler.transform(X_test)               # applique au test
print('Moyenne du test standardisé (pas exactement 0, et c’est normal) :', Z_test.mean(axis=0).round(2))

### ✏️ Exercice 2 — Effet de masque du z-score

Ajoutez trois autres valeurs élevées (30, 32 et 35 mg/L) aux lignes 30 à 32 de `df2`. Combien de valeurs le z-score ($|z|>3$) détecte-t-il ? Et le score robuste ($|z^{rob}|>3{,}5$) ?

In [ ]:
df3 = df2.copy()
df3.loc[[30, 31, 32], col] = [30.0, 32.0, 35.0]
x = df3[col]
z = ...
z_rob = ...
print('z-score  :', (z.abs() > 3).sum(), 'valeur(s) signalée(s)')
print('robuste  :', (z_rob.abs() > 3.5).sum(), 'valeur(s) signalée(s)')

<details><summary><b>Voir le corrigé</b></summary>

```python
df3 = df2.copy()
df3.loc[[30, 31, 32], col] = [30.0, 32.0, 35.0]
x = df3[col]
z = (x - x.mean()) / x.std()
med = x.median()
z_rob = 0.6745 * (x - med) / (x - med).abs().median()
print('z-score  :', (z.abs() > 3).sum(), 'valeur(s) signalée(s)')
print('robuste  :', (z_rob.abs() > 3.5).sum(), 'valeur(s) signalée(s)')
```

La valeur 65 gonfle fortement l’écart-type : les z-scores des valeurs 30 à 35 restent sous 3 et elles ne sont pas signalées (*effet de masque*). La médiane et la MAD sont presque insensibles à ces points : le score robuste les signale toutes les quatre.

</details>

## Interprétation scientifique (à rédiger)

1. Quel mécanisme de données manquantes est le plus plausible dans votre discipline ? Avec quel effet ?
2. Pourquoi la valeur 65 mg/L ne doit-elle pas être supprimée sans vérification ?
3. Comparez les imputations (médiane, régression) : que gagne-t-on, que perd-on ?
4. Rédigez la phrase de « méthodes » qui documenterait votre traitement des données.